# PetFinder: Data Quality and Feature Transformation Pipeline
This notebook addresses the data cleaning (Issue #20) and feature transformation (Issue #21) requirements for the PetFinder Adoption Prediction project.

In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import warnings
warnings.filterwarnings('ignore')

# Note: Upload kaggle.json before running this cell if downloading directly from Kaggle
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c petfinder-adoption-prediction
!unzip -q petfinder-adoption-prediction.zip

Saving kaggle.json to kaggle.json
100% 1.94G/1.94G [00:24<00:00, 84.9MB/s]



In [7]:
# Load the datasets
train_df = pd.read_csv('train/train.csv')
test_df = pd.read_csv('test/test.csv')

print(f"Raw train shape: {train_df.shape}")
print(f"Raw test shape: {test_df.shape}")

Raw train shape: (14993, 24)
Raw test shape: (3972, 23)


## 1. Data Quality Improvements
Resolving missing, duplicated, and inconsistent records[cite: 9].
Based on our exploratory data analysis, no rows are dropped[cite: 4]. The following fixes are applied consistently to both train and test sets:
* **Breeds**: Fixing inverted primary/secondary breeds and redundant secondary breeds[cite: 4].
* **Colors**: Closing gaps where Color2 is missing but Color3 is present[cite: 4].
* **Names**: Replacing placeholder text (e.g., "No Name Yet") with NaN[cite: 4].
* **Descriptions**: Filling missing descriptions with empty strings[cite: 4].
* **Types**: Converting `PhotoAmt` to integer[cite: 4].

In [8]:
def clean_petfinder(df):
    """Applies data quality fixes to the DataFrame."""
    df = df.copy()

    # 1) Fix inverted breeds: Breed1 empty (0) but Breed2 filled
    swap = (df['Breed1'] == 0) & (df['Breed2'] != 0)
    df.loc[swap, 'Breed1'] = df.loc[swap, 'Breed2']
    df.loc[swap, 'Breed2'] = 0

    # 2) Fix redundant secondary breed (Breed2 == Breed1)
    same = (df['Breed2'] == df['Breed1']) & (df['Breed2'] != 0)
    df.loc[same, 'Breed2'] = 0

    # 3) Fix color gaps: Color3 filled while Color2 is empty
    gap = (df['Color2'] == 0) & (df['Color3'] != 0)
    df.loc[gap, 'Color2'] = df.loc[gap, 'Color3']
    df.loc[gap, 'Color3'] = 0

    # 4) Handle placeholder names
    placeholder_pattern = r'no name|noname|unnamed|not named|nameless|unknown'
    name = df['Name'].str.strip()
    df['Name'] = name.mask(name.str.lower().str.contains(placeholder_pattern, na=False))

    # 5) Handle missing descriptions
    df['Description'] = df['Description'].fillna('')

    # 6) Convert PhotoAmt to integer
    df['PhotoAmt'] = df['PhotoAmt'].round().astype(int)

    return df

# Apply cleaning to both datasets
train_clean = clean_petfinder(train_df)
test_clean = clean_petfinder(test_df)
print("Data cleaning applied successfully.")

Data cleaning applied successfully.


## 2. Data Transformation
Transforming raw data into machine-learning-ready formats[cite: 8].
This involves:
* Log transformations for heavily skewed numerical variables (`Age`, `Fee`)[cite: 6].
* Creating binary indicators (`Is_Free`, `Has_Name`, `Is_Mixed`, `Is_Group`)[cite: 6].
* Combining categorical medical features into a single `Medical_Score`[cite: 6].

In [9]:
def process_tabular_features(df):
    """Extracts and transforms tabular features."""
    df = df.copy()

    # Logarithmic transformations
    df['Age_Log'] = np.log1p(df['Age'])
    df['Fee_Log'] = np.log1p(df['Fee'])

    # Binary indicators
    df['Is_Free'] = (df['Fee'] == 0).astype(int)
    df['Has_Name'] = df['Name'].notna().astype(int)

    # Mixed breed indicator (Breed2 is given OR Breed1 is 307/Mixed)
    df['Is_Mixed'] = ((df['Breed2'] > 0) | (df['Breed1'] == 307)).astype(int)

    # Medical condition score (1=Yes, 2=No, 3=Not sure) - counting 'Yes'
    df['Medical_Score'] = (
        (df['Vaccinated'] == 1).astype(int) +
        (df['Dewormed'] == 1).astype(int) +
        (df['Sterilized'] == 1).astype(int)
    )

    # Group indicators and engineered numericals
    df['Is_Group'] = (df['Quantity'] > 1).astype(int)
    df['Fee_Per_Pet'] = df['Fee'] / df['Quantity']

    # Text length stats
    df['Desc_Length'] = df['Description'].apply(len)
    df['Desc_Word_Count'] = df['Description'].apply(lambda x: len(x.split()))

    return df

train_features = process_tabular_features(train_clean)
test_features = process_tabular_features(test_clean)
print("Tabular features processed.")

Tabular features processed.


## 3. Text Representation Pipeline
To avoid data leakage, text transformation (TF-IDF and SVD) is **fitted exclusively on the training data**[cite: 8]. The test data is only transformed using the fitted pipeline[cite: 8].

In [10]:
# Initialize TF-IDF and SVD
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
svd = TruncatedSVD(n_components=10, random_state=42)

# FIT and TRANSFORM on Training Data ONLY
print("Fitting text pipeline on training data...")
train_tfidf = tfidf.fit_transform(train_features['Description'])
train_svd = svd.fit_transform(train_tfidf)

# TRANSFORM on Test Data
print("Transforming test data...")
test_tfidf = tfidf.transform(test_features['Description'])
test_svd = svd.transform(test_tfidf)

# Append SVD components to DataFrames
for i in range(10):
    train_features[f'Desc_SVD_{i}'] = train_svd[:, i]
    test_features[f'Desc_SVD_{i}'] = test_svd[:, i]

print("Text features extracted successfully.")

Fitting text pipeline on training data...
Transforming test data...
Text features extracted successfully.


In [11]:
# Drop original text/id columns that are no longer needed for modeling
cols_to_drop = ['Name', 'Description', 'RescuerID']
train_final = train_features.drop(columns=cols_to_drop)
test_final = test_features.drop(columns=cols_to_drop)

# Export the final feature matrices
train_final.to_csv('train_ready.csv', index=False)
test_final.to_csv('test_ready.csv', index=False)

print(f"Final train shape: {train_final.shape}")
print(f"Final test shape: {test_final.shape}")
print("Ready for feature selection and modeling!")

Final train shape: (14993, 41)
Final test shape: (3972, 40)
Ready for feature selection and modeling!
